# Visualizing stimulus responsiveness

Review units whose firing changes after the first visual stimulus and inspect the shape of each response, including single or multiple peaks and dips. 

Unit classifications and detected response components come from the `StimulusResponsiveness` table. The PETH is recomputed only for display.

#### Set up interactive plotting, import the analysis tools, and choose the session to review.

In [ ]:
%matplotlib widget

import sys
import warnings
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from spks.event_aligned import population_peth

# Silence the setuptools pkg_resources deprecation notice
warnings.filterwarnings("ignore", category=UserWarning, module="datajoint.plugin")

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "labdata_plugin").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from labdata_plugin.schema import (  # noqa: E402
    StimulusResponsiveness,
    StimulusResponsivenessParams,
)
from thesis.ephys.io_digital_events import fetch_session_events  # noqa: E402
from thesis.ephys.io_session_units import fetch_good_units  # noqa: E402

subject = "GRB058"
session = "20260224_152424"
unit_criteria_id = 1
responsiveness_param_id = 0

#### Populate and fetch the stored response classifications, then display the response traces.

In [ ]:
response_key = {
    "subject_name": subject,
    "session_name": session,
    "unit_criteria_id": unit_criteria_id,
    "responsiveness_param_id": responsiveness_param_id,
}
StimulusResponsiveness.populate(response_key)
params = (StimulusResponsivenessParams & response_key).fetch1()
stored_rows = (
    StimulusResponsiveness.Unit & response_key & 'response_type != "none"'
).fetch(
    "unit_id",
    "response_type",
    "n_response_components",
    "response_component_times",
    "response_component_rates",
    as_dict=True,
)
response_rows = pd.DataFrame(stored_rows)

print(f"Loading synchronized traces for {subject} {session}...")
spikes_by_unit = fetch_good_units(subject, session, unit_criteria_id)
first_stimulus = fetch_session_events(subject, session)["first_stim_ev"]
unit_ids = list(spikes_by_unit)
peth, bin_edges, _ = population_peth(
    all_spike_times=list(spikes_by_unit.values()),
    alignment_times=first_stimulus,
    pre_seconds=params["peth_pre_seconds"],
    post_seconds=params["peth_post_seconds"],
    binwidth_ms=params["binwidth_ms"],
)
peth = peth / (params["binwidth_ms"] / 1000)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
baseline_window = (params["baseline_start"], params["baseline_end"])
response_window = (params["response_start"], params["response_end"])

response_counts = response_rows["response_type"].value_counts()
print(
    f"Loaded {len(unit_ids)} units and {len(first_stimulus)} first-stimulus events.\n"
    f"Responsive units: {len(response_rows)} "
    f"({response_counts.get('excited', 0)} excited, "
    f"{response_counts.get('suppressed', 0)} suppressed)."
)

#### Summarize how many peaks or dips were detected in the stimulus-responsive units.

In [ ]:
shape_counts = (
    response_rows.groupby(["n_response_components", "response_type"])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(6, 3.5))
shape_counts.plot.bar(
    ax=ax,
    color={"excited": "#f58518", "suppressed": "#4c78a8"},
    rot=0,
)
ax.set(
    title=f"Response shapes — {subject} {session}",
    xlabel="Detected peaks or dips",
    ylabel="Units",
)
ax.legend(title="Response", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

#### Browse responsive units to inspect the timing and shape of their peaks or dips.

In [ ]:
review_rows = response_rows.sort_values(
    ["response_type", "n_response_components", "unit_id"]
).reset_index(drop=True)
unit_position = {unit_id: position for position, unit_id in enumerate(unit_ids)}
slider_options = []
for index, row in review_rows.iterrows():
    slider_options.append((f"unit {int(row['unit_id'])}", index))

unit_slider = widgets.SelectionSlider(
    options=slider_options,
    description="Unit:",
    continuous_update=False,
    layout=widgets.Layout(width="700px"),
    style={"description_width": "45px"},
)
with plt.ioff():
    review_fig, review_ax = plt.subplots(figsize=(7, 3.5))


def plot_review_unit(index):
    row = review_rows.iloc[index]
    unit_id = int(row["unit_id"])
    mean_peth = peth[unit_position[unit_id]].mean(axis=0)
    baseline_bins = (bin_centers >= baseline_window[0]) & (
        bin_centers < baseline_window[1]
    )
    baseline = mean_peth[baseline_bins].mean()
    response_type = row["response_type"]
    feature_name = "peak" if response_type == "excited" else "dip"
    marker = "v" if response_type == "excited" else "^"
    color = "#f58518" if response_type == "excited" else "#4c78a8"

    review_ax.clear()
    review_ax.plot(bin_centers, mean_peth, color="black", linewidth=1.5)
    review_ax.axvspan(*baseline_window, color="#4c78a8", alpha=0.10)
    review_ax.axvspan(*response_window, color="#f58518", alpha=0.10)
    review_ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    review_ax.axhline(baseline, color="gray", linestyle=":", linewidth=1)
    for feature_time, feature_height in zip(
        row["response_component_times"], row["response_component_rates"]
    ):
        review_ax.plot(feature_time, feature_height, marker, color=color)

    review_ax.set(
        title=(
            f"Unit {unit_id}: {response_type} response with "
            f"{int(row['n_response_components'])} {feature_name}(s)"
        ),
        xlabel="Time from first stimulus (s)",
        ylabel="Firing rate (sp/s)",
    )
    review_ax.spines[["top", "right"]].set_visible(False)
    review_fig.tight_layout()
    review_fig.canvas.draw_idle()


def on_unit_change(change):
    plot_review_unit(change["new"])


unit_slider.observe(on_unit_change, names="value")
plot_review_unit(unit_slider.value)
display(unit_slider)
display(review_fig.canvas)